# Introducción

La compañía OilyGiant busca determinar cuál de tres regiones disponibles ofrece las mejores condiciones para el desarrollo de 200 nuevos pozos petrolíferos. Para ello, se dispone de datos geológicos que incluyen características de los pozos y el volumen de reservas de petróleo presentes en cada región. A partir de esta información se desarrollará un modelo de machine learning que permita estimar las reservas de nuevos pozos y apoyar la toma de decisiones de inversión.

Además de analizar la rentabilidad potencial de cada región, es necesario evaluar los riesgos asociados a la inversión. La empresa cuenta con un presupuesto de 100 millones de dólares y únicamente considerará regiones cuyo riesgo de pérdidas sea inferior al 2.5%, por lo que la decisión final deberá considerar tanto el beneficio esperado como la incertidumbre de los resultados.

Para cumplir con este objetivo, el proyecto se desarrollará mediante las siguientes etapas:

1. Carga y exploración de los datos correspondientes a las tres regiones de estudio.
2. Entrenamiento y evaluación de modelos de regresión lineal para estimar el volumen de reservas de petróleo en cada región.
3. Preparación de los parámetros necesarios para el cálculo de beneficios y análisis del punto de equilibrio del proyecto.
4. Cálculo de la ganancia potencial mediante la selección de los 200 pozos con las mayores reservas estimadas.
5. Aplicación de la técnica de bootstrapping para generar distribuciones de ganancias para cada región.
6. Cálculo del beneficio promedio, intervalos de confianza y riesgo de pérdidas asociados a cada alternativa.
7. Comparación de los resultados obtenidos y selección de la región más conveniente para el desarrollo de nuevos pozos petrolíferos.

Finalmente, se presentará una recomendación basada en criterios de rentabilidad y riesgo, identificando la región que ofrece las mejores condiciones para la inversión de la compañía.

## 1. Importación de librerías

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

## 2. Cargar los datos

In [2]:
geo_0 = pd.read_csv('/datasets/geo_data_0.csv')
geo_1 = pd.read_csv('/datasets/geo_data_1.csv')
geo_2 = pd.read_csv('/datasets/geo_data_2.csv')

In [3]:
display(geo_0.head())
display(geo_1.head())
display(geo_2.head())

,id,f0,f1,f2,product
0,txEyH,0.705745,-0.497823,1.221170,105.280062
1,2acmU,1.334711,-0.340164,4.365080,73.037750
2,409Wp,1.022732,0.151990,1.419926,85.265647
3,iJLyR,-0.032172,0.139033,2.978566,168.620776
4,Xdl7t,1.988431,0.155413,4.751769,154.036647


,id,f0,f1,f2,product
0,kBEdx,-15.001348,-8.276000,-0.005876,3.179103
1,62mP7,14.272088,-3.475083,0.999183,26.953261
2,vyE1P,6.263187,-5.948386,5.001160,134.766305
3,KcrkZ,-13.081196,-11.506057,4.999415,137.945408
4,AHL4O,12.702195,-8.147433,5.004363,134.766305


,id,f0,f1,f2,product
0,fwXo0,-1.146987,0.963328,-0.828965,27.758673
1,WJtFt,0.262778,0.269839,-2.530187,56.069697
2,ovLUW,0.194587,0.289035,-5.586433,62.871910
3,q6cA6,2.236060,-0.553760,0.930038,114.572842
4,WPMUX,-0.515993,1.716266,5.899011,149.600746


In [4]:
print(geo_0.info())
print(geo_1.info())
print(geo_2.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 5 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   id       100000 non-null  object 
 1   f0       100000 non-null  float64
 2   f1       100000 non-null  float64
 3   f2       100000 non-null  float64
 4   product  100000 non-null  float64
dtypes: float64(4), object(1)
memory usage: 3.8+ MB
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 5 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   id       100000 non-null  object 
 1   f0       100000 non-null  float64
 2   f1       100000 non-null  float64
 3   f2       100000 non-null  float64
 4   product  100000 non-null  float64
dtypes: float64(4), object(1)
memory usage: 3.8+ MB
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 5 columns):
 #   Column  

Los datos contienen información de tres regiones distintas. Cada archivo incluye un identificador único del pozo, tres características numéricas y la variable objetivo `product`, que representa el volumen de reservas en miles de barriles.

Para el entrenamiento del modelo no se utilizará la columna `id`, ya que solo identifica cada pozo y no aporta información predictiva directa.

## 3. Entrenamiento y validación del modelo

In [5]:
def train_model(data):
    features = data.drop(['id', 'product'], axis=1)
    target = data['product']
    
    features_train, features_valid, target_train, target_valid = train_test_split(
        features,
        target,
        test_size=0.25,
        random_state=12345
    )
    
    model = LinearRegression()
    model.fit(features_train, target_train)
    
    predictions = model.predict(features_valid)
    
    predictions = pd.Series(predictions, index=target_valid.index)
    target_valid = pd.Series(target_valid, index=target_valid.index)
    
    rmse = mean_squared_error(target_valid, predictions) ** 0.5
    mean_prediction = predictions.mean()
    
    return predictions, target_valid, mean_prediction, rmse

In [6]:
pred_0, target_0, mean_pred_0, rmse_0 = train_model(geo_0)
pred_1, target_1, mean_pred_1, rmse_1 = train_model(geo_1)
pred_2, target_2, mean_pred_2, rmse_2 = train_model(geo_2)

In [7]:
results = pd.DataFrame({
    'Región': ['Región 0', 'Región 1', 'Región 2'],
    'Reservas medias predichas': [mean_pred_0, mean_pred_1, mean_pred_2],
    'RMSE': [rmse_0, rmse_1, rmse_2]
})

display(results)

,Región,Reservas medias predichas,RMSE
0,Región 0,92.592568,37.579422
1,Región 1,68.728547,0.893099
2,Región 2,94.965046,40.029709


La Región 1 muestra un RMSE de 0.89, prácticamente nulo en comparación con el de la Región 0 (37.58) y la Región 2 (40.03). Esto indica que el modelo predice el volumen de reservas en la Región 1 con una precisión casi perfecta, mientras que en las otras dos regiones el error promedio es muy alto en relación con la propia escala de `product` (que ronda 92-95 unidades de media).

Aunque la Región 0 (92.59) y la Región 2 (94.97) presentan reservas medias predichas más altas que la Región 1 (68.73), ese resultado es poco confiable dado su elevado error de predicción: el modelo apenas logra capturar la relación entre las features y el target en esas dos regiones. En la Región 1, en cambio, la combinación de baja reserva media pero error casi nulo sugiere que las predicciones son mucho más fiables para seleccionar los mejores pozos en el siguiente paso.

## 4. Preparación para el cálculo de ganancias

In [8]:
BUDGET = 100_000_000
WELLS_TOTAL = 500
WELLS_SELECTED = 200
REVENUE_PER_UNIT = 4500

BUDGET_PER_WELL = BUDGET / WELLS_SELECTED
MIN_PRODUCT_PER_WELL = BUDGET_PER_WELL / REVENUE_PER_UNIT

print('Presupuesto por pozo:', BUDGET_PER_WELL)
print('Reservas mínimas necesarias por pozo:', MIN_PRODUCT_PER_WELL)

Presupuesto por pozo: 500000.0
Reservas mínimas necesarias por pozo: 111.11111111111111


In [9]:
mean_products = pd.DataFrame({
    'Región': ['Región 0', 'Región 1', 'Región 2'],
    'Reserva media real': [
        geo_0['product'].mean(),
        geo_1['product'].mean(),
        geo_2['product'].mean()
    ],
    'Reserva mínima necesaria': [
        MIN_PRODUCT_PER_WELL,
        MIN_PRODUCT_PER_WELL,
        MIN_PRODUCT_PER_WELL
    ]
})

display(mean_products)

,Región,Reserva media real,Reserva mínima necesaria
0,Región 0,92.500,111.111111
1,Región 1,68.825,111.111111
2,Región 2,95.000,111.111111


Para no tener pérdidas, cada pozo debe producir en promedio al menos 111.11 miles de barriles (la reserva mínima necesaria, calculada como presupuesto por pozo / ingreso por unidad).

Al comparar este umbral con la reserva media real de cada región, se observa que ninguna de las tres regiones alcanza ese mínimo si se seleccionaran pozos al azar: la Región 0 tiene una media real de 92.50, la Región 1 de 68.83 y la Región 2 de 95.00, todas por debajo de 111.11. Esto confirma que invertir en una muestra aleatoria de pozos generaría pérdidas en cualquiera de las tres regiones.

Por esta razón, el paso siguiente es fundamental: no se deben elegir pozos al azar, sino seleccionar específicamente los 200 pozos con las predicciones más altas dentro de cada muestra de 500, ya que solo así es posible superar el punto de equilibrio y obtener ganancias.

## 5. Función para calcular ganancias

In [10]:
def profit(target, predictions, count):
    predictions_sorted = predictions.sort_values(ascending=False)
    selected = target[predictions_sorted.index][:count]
    
    revenue = selected.sum() * REVENUE_PER_UNIT
    profit_value = revenue - BUDGET
    
    return profit_value

In [11]:
profit_0 = profit(target_0, pred_0, WELLS_SELECTED)
profit_1 = profit(target_1, pred_1, WELLS_SELECTED)
profit_2 = profit(target_2, pred_2, WELLS_SELECTED)

profit_results = pd.DataFrame({
    'Región': ['Región 0', 'Región 1', 'Región 2'],
    'Ganancia potencial': [profit_0, profit_1, profit_2]
})

display(profit_results)

,Región,Ganancia potencial
0,Región 0,3.320826e+07
1,Región 1,2.415087e+07
2,Región 2,2.710350e+07


Al seleccionar los 200 pozos con las predicciones más altas de cada región y calcular la ganancia con sus valores reales correspondientes, se obtuvieron los siguientes resultados:

- **Región 0:** ganancia potencial de 33.21 millones de dólares.
- **Región 1:** ganancia potencial de 24.15 millones de dólares.
- **Región 2:** ganancia potencial de 27.10 millones de dólares.

En este escenario ideal (validación completa, sin muestreo), la Región 0 parece la más rentable. Sin embargo, este cálculo usa todo el conjunto de validación como si fuera la muestra de exploración, lo que no refleja la realidad del negocio: la empresa estudia solo 500 puntos por región y elige los 200 mejores dentro de esa muestra reducida. Por eso esta cifra no es suficiente para tomar la decisión final. Es necesario aplicar bootstrapping para simular ese escenario realista y, sobre todo, para cuantificar el riesgo de pérdidas, ya que la Región 0 tenía el RMSE más alto y por tanto sus predicciones son las menos confiables.

## 6. Bootstrapping para riesgos y ganancias

In [16]:
def bootstrap_profit(target, predictions):
    state = np.random.RandomState(12345)
    values = []
    
    for i in range(1000):
        target_subsample = target.sample(
            n=WELLS_TOTAL,
            replace=True,
            random_state=state
        )
        
        predictions_subsample = predictions[target_subsample.index]
        
        profit_value = profit(
            target_subsample,
            predictions_subsample,
            WELLS_SELECTED
        )
        
        values.append(profit_value)
    
    values = pd.Series(values)
    
    mean_profit = values.mean()
    lower = values.quantile(0.025)
    upper = values.quantile(0.975)
    risk = (values < 0).mean() * 100
    
    return mean_profit, lower, upper, risk

In [17]:
mean_0, lower_0, upper_0, risk_0 = bootstrap_profit(target_0, pred_0)
mean_1, lower_1, upper_1, risk_1 = bootstrap_profit(target_1, pred_1)
mean_2, lower_2, upper_2, risk_2 = bootstrap_profit(target_2, pred_2)

In [18]:
bootstrap_results = pd.DataFrame({
    'Región': ['Región 0', 'Región 1', 'Región 2'],
    'Ganancia promedio': [mean_0, mean_1, mean_2],
    'IC 95% inferior': [lower_0, lower_1, lower_2],
    'IC 95% superior': [upper_0, upper_1, upper_2],
    'Riesgo de pérdidas (%)': [risk_0, risk_1, risk_2]
})

display(bootstrap_results)

,Región,Ganancia promedio,IC 95% inferior,IC 95% superior,Riesgo de pérdidas (%)
0,Región 0,4.259385e+06,-1.020901e+06,9.479764e+06,6.0
1,Región 1,5.152228e+06,6.887323e+05,9.315476e+06,1.0
2,Región 2,4.350084e+06,-1.288805e+06,9.697070e+06,6.4


## 7. Conclusión final

Tras aplicar bootstrapping con 1000 muestras de 500 pozos cada una, se observó que la Región 1 presentó la mayor ganancia promedio, aproximadamente 5.15 millones de dólares, además de un riesgo de pérdidas del 1.0%. Por su parte, la Región 0 obtuvo una ganancia promedio cercana a 4.26 millones de dólares y un riesgo de pérdidas del 6.0%, mientras que la Región 2 registró una ganancia promedio de aproximadamente 4.35 millones de dólares y un riesgo del 6.4%.

La condición de negocio establece que únicamente pueden considerarse regiones con un riesgo de pérdidas inferior al 2.5%. Bajo este criterio, las regiones 0 y 2 deben descartarse debido a que superan ampliamente el nivel de riesgo permitido. En contraste, la Región 1 es la única que cumple con esta restricción, lo que la convierte en la única alternativa viable para la inversión.

Es importante destacar que este resultado no coincide con la evaluación preliminar realizada a partir de la ganancia potencial de los 200 mejores pozos. Aunque la Región 0 mostraba inicialmente la mayor ganancia potencial, dicho análisis no contemplaba la incertidumbre asociada a las predicciones ni la variabilidad presente en muestras reales de 500 pozos. Al incorporar estos factores mediante la técnica de bootstrapping, se evidencia que la Región 0 presenta un nivel de riesgo demasiado elevado para satisfacer las condiciones establecidas por la empresa.

Teniendo en cuenta todo lo previamente expuesto, se recomienda desarrollar los 200 nuevos pozos petrolíferos en la Región 1. Esta región no solo cumple con el requisito de mantener un riesgo de pérdidas inferior al 2.5%, sino que además presenta la mayor ganancia promedio esperada entre las tres alternativas analizadas, ofreciendo el mejor equilibrio entre rentabilidad y seguridad para la inversión.